# Chapter 10 — Distributed Data Parallel Training

Training GPT-3 (175B parameters) on a single GPU would take decades. Modern LLM training requires distributing computation across hundreds or thousands of GPUs working in concert. This chapter covers the parallelism strategies that make large-scale training possible, from the straightforward to the highly sophisticated.

In [ ]:
import torch
import torch.nn as nn

world_size = 4
batch_per_gpu = 8

print("=== Distributed Data Parallel Simulation ===")
print(f"World size (GPUs):    {world_size}")
print(f"Batch per GPU:        {batch_per_gpu}")
print(f"Effective batch size: {batch_per_gpu * world_size}\n")

torch.manual_seed(42)
template = nn.Linear(8, 1, bias=False)

gpu_grads = []
for rank in range(world_size):
    x = torch.randn(batch_per_gpu, 8)
    y = torch.randn(batch_per_gpu, 1)
    model = nn.Linear(8, 1, bias=False)
    model.weight.data.copy_(template.weight.data)
    loss = ((model(x) - y) ** 2).mean()
    loss.backward()
    gpu_grads.append(model.weight.grad.clone())
    print(f"  Rank {rank}: loss={loss.item():.4f}, grad_norm={model.weight.grad.norm():.4f}")

avg_grad = torch.stack(gpu_grads).mean(dim=0)
print(f"\nAfter All-Reduce (gradient averaging):")
print(f"  Averaged grad norm: {avg_grad.norm():.4f}")
print(f"  All GPUs apply the SAME gradient -> models stay in sync")

In [ ]:
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def train(rank, world_size):
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    model = GPT(config).cuda(rank)
    model = DDP(model, device_ids=[rank])
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, sampler=sampler, batch_size=32)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    for epoch in range(num_epochs):
        sampler.set_epoch(epoch)
        for batch in dataloader:
            loss = model(batch.cuda(rank)).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    dist.destroy_process_group()

---

**Course:** Aprenda LLM | **Chapter 10**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.